CBAM fusion in this file,
Contains architectural decisions made by harshil, dont change this

In [ ]:
import torch
import torch.nn.functional as nn

In [ ]:
class CBAMFusion(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn_noise = ChannelAttention(channels, reduction)
        self.spatial_attn_noise = SpatialAttention(kernel_size)
        self.channel_attn_edge = ChannelAttention(channels, reduction)
        self.spatial_attn_edge = SpatialAttention(kernel_size)
        self.fuse_conv = nn.Conv2d(2 * channels, channels, kernel_size=1)

    def forward(self, noise_feat, edge_feat):
        # noise_feat, edge_feat: both [B, C, H, W]

        n = self.channel_attn_noise(noise_feat)
        n = self.spatial_attn_noise(n)

        e = self.channel_attn_edge(edge_feat)
        e = self.spatial_attn_edge(e)

        combined = torch.cat([n, e], dim=1)   # [B, 2C, H, W]
        fused = self.fuse_conv(combined)      # [B, C, H, W]

        return fused